# Overview

The n8n (pronounced n-eight-n) solution is a workflow automation tool with native support for AI. Like many of it's competitors, it represents workflows as as graphs which are configured through a no or low code drag and drop canvas. It is provided implemented as a node.js executable and does have the ability to horizontally scale.

# Licensing and Commercial Support

After looking at the site, it appears there are several tiers of products being offered:
* Comunity Edition - A "source available" basic version provided on github
* A Hosted Solution (with starter, pro, and enterprise subscriptions) - offering varying SLAs and support.
* A Self-hosted Enterprise Subscription
* A Startup Plan - Avaiable to startups with up to 20 employees which raised up to $5M.

According to the [LICENSE.md](https://github.com/n8n-io/n8n/blob/master/LICENSE.md) file in the github repository, the source code is distributed under several licences:
* Content of branches other than the main branch (i.e. "master") are not licensed.
* All source code files that contain ".ee." in their filename are licensed under the "n8n Enterprise License" defined in "LICENSE_EE.md".
* All third party components incorporated into the n8n Software are licensed under the original license provided by the owner of the applicable component.
* Content outside of the above mentioned files or restrictions is available under the "Sustainable Use License" (defined within this file)

# Solution Architecture

The n8n solution is provided as an executable which runs on node.js. As such it is packaged and distributed by npm and npx. The official github page can be found [here](https://github.com/n8n-io/n8n).

As all of the Workflows and Nodes are defined as TypeScript classes, they are orchestrated and executed within the singular binary. This means that n8n can be deployed in any traditional format: On prem, VM, Docker, k8s, and more.

## Scalability

The [documentation](https://docs.n8n.io/hosting/scaling/queue-mode/) notes a master-slave configuration called "queue-mode" is possible. It states:

> When running in queue mode, you have multiple n8n instances set up, with one main instance receiving workflow information (such as triggers) and the worker instances performing the executions.
>
> Each worker is its own Node.js instance, running in main mode, but able to handle multiple simultaneous workflow executions due to their high IOPS (input-output operations per second).
>
> By using worker instances and running in queue mode, you can scale n8n up (by adding workers) and down (by removing workers) as needed to handle the workload.

Reading on, we see that Redis maintains the work queue and a database (such as postgres) records the results. The master and slaves communicate through these two data stores. Additionally, n8n does not provide mechanisms for deploying this configuration or the required datastores; that is left to the engineer.

# Major Concepts

## Workflow

A workflow is a logical collection of related Nodes. Conceptually, the workflow describes an end-to-end process that is articulated by one or more Nodes. The Workflows are implimented as graphs of Nodes and visualized on a drag-and-drop canvas.

## Workflow Templates

As the name suggests, a Workflow template is a reusable workflow definition, which consists of a preconfigured set of Nodes which a user can instantiate and configure to produce a fully functional workflow without having to design it from scratch. 

The [integration page](https://n8n.io/integrations/) lists a number of example templates which can be downloaded and installed.

## Nodes

According to the [documentation ](https://docs.n8n.io/workflows/components/nodes), Nodes are the key building blocks of a workflow. The Nodes divide the workflow into smaller pieces and abstract away the underlying implementation details. n8n provides a collection of built-in nodes, as well as the ability to download and install comunity nodes, and create your own nodes. 

Nodes are implemented as TypeScript classes which extend the INodeType interface. There are two styles (declarative and programatic) for defining a node as described [here](https://docs.n8n.io/integrations/creating-nodes/plan/choose-node-method/#syntax-differences). Additionally, n8n provides libraries for defining the UI elements of the Nodes.

As the nodes are defined as TypeScript classes, they can be packaged and distributed through npm as described [here](https://www.npmjs.com/package/n8n-node-dev).

### LangChain Integration

The Nodes provide the user with the ability to leverage LangChain functionality. As the Nodes are implimented in javscript, the LangChain integration happens through the javascript LangChain library. There are several Nodes which n8n offers to expose LangChain functionality, for more information review this [documentation](https://docs.n8n.io/advanced-ai/langchain/langchain-n8n/).

## Connections

According to the [documentation](https://docs.n8n.io/workflows/components/connections/):

> A connection establishes a link between nodes to route data through the workflow. A connection between two nodes passes data from one node's output to another node's input.

## Logical Operators

The n8n solution offers a number of [means for expressing logical patterns](https://docs.n8n.io/flow-logic/) within a Workflow. Conditional splitting, Looping, Waiting, Error Handling, and more are all provided as off-the-shelf functionality which is exposed via the Nodes.

## Sticky Notes

According to the [documentation](https://docs.n8n.io/workflows/components/sticky-notes/), Sticky Notes allow you to annotate and comment on your workflows by visually placing a colored box of text within the workflow canvas. Sticky Notes support the use of Markdown.

## Tags

Within n8n, [tags](https://docs.n8n.io/workflows/tags/) allow the user to attach globally available labels to workflows. These labels can then be used when querying the system to filter based on the Tag.

## Credentials

[Credentials](https://docs.n8n.io/credentials/) allow n8n users to store the various bits of information used to authenticate with external systems. Users can create and share credentials with other users. The credentials can be linked to Nodes, giving the node the ability to authenticate while interacting with an external system.

## Users & Account Types

When a user is created, the User is assigned an account type. There are three account types, owner, admin, and member. The account type affects the user permissions and access.For more information on account types, see this [documentation](https://docs.n8n.io/user-management/account-types/).

# Security Features

The solution offers two factor authentication, LDAP integration, and SAML SSO. For more details consult the [documentation](https://docs.n8n.io/user-management/).

# VCS Integration

Git is integrated directly into n8n as decribed [here](https://docs.n8n.io/source-control-environments/understand/git/), however as noted in the documentation:

> n8n doesn't implement all Git functionality: you shouldn't view n8n's source control as full version control.

This integration allows users to edit their workflows in the n8n web interface while having them versioned and stored in a git provider. It appears that n8n allows for committing, pushing and pulling. The rest of the git workflow will need to be executed on the backend VCS.

Looking at the [screenshots](https://docs.n8n.io/source-control-environments/using/push-pull), it appears that the UI provides built in controls for pushing and pulling a workflow to a branch. However, n8n recommends only establishing a unidirectional workflow:

> You can push work from an instance to a branch, and pull to the same instance. n8n doesn't recommend this. To reduce the risk of merge conflicts and overwriting work, try to create a process where work goes in one direction: either to Git, or from Git, but not both.



Another important consideration when using the git integration is the effect the push/pull events have on the workflow:

> If you pull changes to an active workflow, n8n sets the workflow to inactive while pulling, then reactivates it. This may result in a few seconds of downtime for the workflow.

# REST API

While the documentation mentions there is a REST API, it is not clear if this is feature complete (i.e. anything possible in the UI is possible via the API). Looking through the [documentation](https://docs.n8n.io/api) it appears that creating Users and Workflows is possible as well as triggering a workflow.

# External Hooks

We [see](https://docs.n8n.io/embed/configuration/#frontend-external-hooks) that n8n allows us to execute webhooks when specific actions occur on the frontend and the backend:

> Like backend external hooks, it's possible to define external hooks in the frontend code that get executed by n8n whenever a specific operation is performed. They can be used, for example, to log data and change data.